In [34]:
import openai
import json
from tqdm import tqdm
from collections import defaultdict
import sys
sys.path.append("/data2/polyakov/instruction_tuning/ToolBench")
openai.api_key_path = '/data4/polyakov/GPT4_KEY_NEW'

In [35]:
def make_gpt3_request(prompt: str, max_tokens=100):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
                {"role": "system", "content": "You are a helpful assistant, useful for code generation."},
                {"role": "user", "content": prompt},
            ],
        stop=None,
        max_tokens=max_tokens
        )
    return response

In [36]:
# r = openai.api_requestor.APIRequestor()
# resp = r.request("GET", '/dashboard/billing/usage?end_date=2023-11-17&start_date=2023-11-14') #or start_date and end_date
# resp_object = resp[0]
# resp_object.data

In [37]:
r = openai.api_requestor.APIRequestor()
resp = r.request("GET", '/usage?date=2023-11-18');
resp_object = resp[0]
resp_object.data

{'object': 'list',
 'data': [],
 'ft_data': [],
 'dalle_api_data': [],
 'whisper_api_data': []}

In [38]:
52 * 0.001

0.052000000000000005

In [39]:
len(resp_object.data['data'])

0

In [40]:
make_gpt3_request('How are you?')

<OpenAIObject chat.completion id=chatcmpl-8NdDpt0OKNBjqJZyyjkWrsk1rOr1h at 0x7f20585e0d10> JSON: {
  "id": "chatcmpl-8NdDpt0OKNBjqJZyyjkWrsk1rOr1h",
  "object": "chat.completion",
  "created": 1700643005,
  "model": "gpt-3.5-turbo-0613",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "Thank you for asking! I'm an AI assistant designed to provide coding assistance. I'm here to help you with any questions or code generation requests you may have. How can I assist you today?"
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 26,
    "completion_tokens": 40,
    "total_tokens": 66
  }
}

In [53]:
errors = json.load(open('responses_message_error.json', 'r'))
idx = 1
query = errors[idx]['answer_generation']['query']
error_chain_example = errors[idx]['trys'][0]['chain'][:3]
tool_descriptions = str(errors[idx]['answer_generation']['function'])
errors[idx]

{'win': False,
 'try_count': 1,
 'trys': [{'chain': [{'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 1,
     'node_type': 'Thought',
     'description': '',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 2,
     'node_type': 'Action',
     'description': 'get_distance_by_city_state_country_for_great_circle_math_api',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 3,
     'node_type': 'Action Input',
     'description': '{\n  "country1": "US",\n  "state1": "CA",\n  "city1": "Escondido",\n  "country2": "US",\n  "state2": "CA",\n  "city2": "Santa Rosa"\n}',
     'Elo': 1000.0,
     'observation': '{"error": "", "response": "{\'cityName1\': \'Escondido\', \'stateName1\': \'California\', \'country1\': \'US\', \'latitude1\': 33.1216751, \'longitude1

In [55]:
prompt = ''
for item in error_chain_example:
    if item['node_type'] != 'Action Input':
        prompt += item['node_type'] + ': ' + item['description'] + '\n'
    else:
        prompt += item['node_type'] + ': ' + item['description'] + '\n' + 'Observation: ' + item['observation'] + '\n'

request = f'The user asked: {query}\n'
tool_descriptions_prompt = f'Here are tool descriptions: {tool_descriptions}\n'
initial_prompt = 'You are given an example of tool call by a model called Toollama and a response from a tool.\nExample begins\n'
final_prompt = tool_descriptions_prompt + request + initial_prompt + prompt + 'Example ends\nAnswer, if this call resulted in an error. If so, explain an error. Do not write any code. Do not make up anything. Explain an error based on tool description, tool parameters format (in the field "parameters"), on response from tool and on your knowledge. Make your response short, just single sentence.\n'
print(final_prompt)

Here are tool descriptions: [{'name': 'get_distance_by_city_state_country_for_great_circle_math_api', 'description': 'This is the subfunction for tool "great_circle_math_api", you can use this tool.The description of this function is: "Takes city, state, and country of both locations and returns latitude, longitude, and calculated miles."', 'parameters': {'type': 'object', 'properties': {'country1': {'type': 'string', 'description': '', 'example_value': 'us'}, 'country2': {'type': 'string', 'description': '', 'example_value': 'us'}, 'state2': {'type': 'string', 'description': '', 'example_value': 'ca'}, 'city2': {'type': 'string', 'description': '', 'example_value': 'sacramento'}, 'city1': {'type': 'string', 'description': '', 'example_value': 'birmingham'}, 'state1': {'type': 'string', 'description': '', 'example_value': 'al'}}, 'required': ['country1', 'country2', 'state2', 'city2', 'city1', 'state1'], 'optional': []}}, {'name': 'm_one_unit_of_measure_to_another_for_measurement_units

Посмотрим, как модель справится со сложной ошибкой из thirty tools

In [56]:
thirty_tools_error_text = open('thirty_tools_errors_examples_1.txt', 'r').read()

In [57]:
print(thirty_tools_error_text)

GPT4Tools can handle various tasks. 
It generates human-like text and uses tools to follow user instructions.
To call API tool it writes python code according to the API tool's description.

TOOLS:
------

GPT4Tools has access to the following API tools:

API Name: pintapi_convert_units
API Parameter: from_value: int or float, amount of something measured. from_unit: string: unit to convert from. to_unit: string: unit to convert to.
API Description: Convert from one unit to another.
Usage Example: Hey, how many kilometers are there in 25 miles?
```python
pintapi_convert_units(from_value=25, from_unit=\"miles\", to_unit=\"kilometers\")
```

To use a tool, please use the following format:

```
Thought: Do I need to use a tool? Yes
Thought: Which tool should I use? the action to take, should be one of the API tools 
AI: python code according to the API tool's description, including python function with the exact same name as the action name and it's parameters
Observation: the result of t

In [58]:
test_prompt = 'You will be given prompt and output of a model called GPT4Tools. If there is an error in a tool call, please describe it breifly.\n'
test_prompt += thirty_tools_error_text
test_prompt += "\nDo not perform tool call. Do not follow GPT4Tools rules. Just explain briefly an error briefly in one sentence."

In [50]:
make_gpt3_request(test_prompt)

RateLimitError: Your account is not active, please check your billing details on our website.

Теперь попробуем запрепроцессить и подать полностью цепочку, до момента которой модель сделала ошибку

In [3]:
from preprocess.preprocess_toolllama_data import preprocess_rapidapi

/data2/polyakov/anaconda3/envs/alpaca-lora/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def preprocess_data_into_prompt

answer_generation = data_dict["answer_generation"]
        is_valid = answer_generation["valid_data"]
        if not is_valid:
            continue
        train_messages = answer_generation["train_messages"]
        query = answer_generation["query"]
        functions = answer_generation["function"]
        for train_message in train_messages:
            conversations = []
            cur_react = ""
            for message_id, message_dict in enumerate(train_message):
                role = message_dict["role"]
                content = message_dict["content"]
                if role == "assistant":
                    inputs = process_assistant_reply(message_dict) 
                    
                    # process the last assistant message as target
                    if message_id + 1 == len(train_message):
                        if "function_call" not in message_dict:
                            cur_react = ""
                            break
                        else:
                            if cur_react == "":
                                cur_react += "\nThought: "
                            action = inputs["name"]
                            action_input = inputs["arguments"]
                            cur_react += f"\nAction: {action}"
                            cur_react += f"\nAction Input: {action_input}"
                            conversations.append({
                                "from": role,
                                "value": cur_react
                            })
                            cur_react = ""
                        tmp_dict = {
                            "id": f"Step {str(message_id)}: {query}",
                            "conversations":conversations
                        }
                        tmp_instances.append(tmp_dict)      
                        break

                    # process the former assistant messages into history conversations
                    else:
                        if "function_call" not in message_dict:
                            cur_react += f"\nThought: {inputs}"
                            continue
                        else:
                            if cur_react == "":
                                cur_react += "\nThought: "
                            action = inputs["name"]
                            action_input = inputs["arguments"]
                            cur_react += f"\nAction: {action}"
                            cur_react += f"\nAction Input: {action_input}"
                            conversations.append({
                                "from": role,
                                "value": cur_react
                            })
                            cur_react = ""
                else:
                    if role == "system":
                        inputs = process_system_message(content, functions)
                    else:
                        inputs = content
                    conversations.append({
                        "from": role,
                        "value": inputs
                    })
                    cur_react = ""